# Beginner E2: classify accents with ECAPA-TDNN embeddings

This is a step-by-step companion to `E1_01_svm_beginner.ipynb`, `E1_04_ivector_beginner.ipynb`, and `E2_03_xvector_beginner.ipynb`. It uses the same `Singlish/data/audio` speaker folders.

**Workflow:** short speech clip → pretrained ECAPA-TDNN speaker model → one embedding → small accent classifier → evaluation on unseen speakers.

ECAPA-TDNN is a speaker-embedding network. It combines temporal convolution blocks, residual connections, channel attention, and **attentive statistics pooling** to turn variable-length speech into one fixed-length vector. We reuse a pretrained VoxCeleb model and train only the accent classifier. This is a transfer-learning experiment, **not** training ECAPA-TDNN from scratch.

This notebook lets you compare ECAPA-TDNN embeddings with the older x-vector architecture on the same task. MNSC and Common Voice also differ in recording conditions and prompts, so a high score alone does not establish accent recognition.


## Before you start

In Colab, choose **Runtime → Change runtime type → T4 GPU** if one is available. The pretrained embedding step can use it; the logistic-regression classifier runs on CPU. A CPU runtime also works, especially for a 100-clips-per-class learning run.

The first extraction downloads the public [`speechbrain/spkrec-ecapa-voxceleb`](https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb) checkpoint. It was trained on VoxCeleb 1 and 2 for speaker recognition, not on this project's accent labels.


In [ ]:
# Colab already supplies PyTorch. SpeechBrain loads the pretrained ECAPA-TDNN embedding model.
%pip -q install "speechbrain>=1.0.3,<2" "librosa>=0.10.2" scikit-learn matplotlib joblib soundfile


## 1. Point to the same audio folders as E1

Put clips in `Singlish/data/audio/singapore/<speaker>/` and `Singlish/data/audio/non_singapore/<speaker>/`. A speaker folder can contain several recordings. Begin with 100 clips per class, then set `MAX_CLIPS_PER_CLASS = None` for all clips. Set `USE_GOOGLE_DRIVE = False` and change `PROJECT_ROOT` when running locally.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ecapa_tdnn'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
MAX_CLIPS_PER_CLASS = 100  # None uses every clip.
SAMPLE_RATE = 16_000
MAX_SECONDS = 6
MODEL_NAME = 'speechbrain/spkrec-ecapa-voxceleb'
print('Audio:', DATA_ROOT)
print('Output:', OUTPUT_DIR)


In [ ]:
import json
from collections import Counter

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from speechbrain.inference.speaker import EncoderClassifier

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Embedding device:', DEVICE)


## 2. Find the recordings and their speakers

The labels are `0 = non-Singapore` and `1 = Singapore`. Selecting clips round-robin across speaker folders reduces the chance that a small learning subset is dominated by a few voices. The speaker ID is kept as a group for the later split.


In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}

def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            parts = path.relative_to(class_dir).parts
            if len(parts) < 2:
                raise ValueError(f'Put this clip inside a speaker folder: {path}')
            rows.append((path, label, f'{class_name}::{parts[0]}'))
    return rows

def select_recordings(rows, limit, seed):
    if limit is None:
        return rows
    if not isinstance(limit, int) or isinstance(limit, bool) or limit < 1:
        raise ValueError('MAX_CLIPS_PER_CLASS must be a positive integer or None')
    selection_rng = np.random.default_rng(seed)
    chosen = []
    for label in CLASS_LABELS.values():
        by_speaker = {}
        for row in rows:
            if row[1] == label:
                by_speaker.setdefault(row[2], []).append(row)
        speakers = sorted(by_speaker)
        selection_rng.shuffle(speakers)
        for speaker in speakers:
            selection_rng.shuffle(by_speaker[speaker])
        class_count = 0
        while class_count < limit:
            added = False
            for speaker in speakers:
                if by_speaker[speaker]:
                    chosen.append(by_speaker[speaker].pop())
                    class_count += 1
                    added = True
                    if class_count == limit:
                        break
            if not added:
                break
    return sorted(chosen, key=lambda row: str(row[0]))

recordings = select_recordings(find_recordings(DATA_ROOT), MAX_CLIPS_PER_CLASS, RANDOM_SEED)
if not recordings:
    raise RuntimeError(f'No audio found below {DATA_ROOT}')
print('Clips:', Counter(row[1] for row in recordings))
print('Speakers:', {label: len({row[2] for row in recordings if row[1] == label}) for label in (0, 1)})


## 3. Prepare audio for the pretrained network

SpeechBrain's `encode_batch` expects a **mono, 16 kHz waveform**. Librosa resamples and mixes channels, trims outer silence, and caps speech at six seconds. The pretrained model computes its own acoustic features, processes them with ECAPA-TDNN blocks, and uses attentive statistics pooling to produce a fixed-length embedding. We do not hand the accent label to the network.

`encode_batch` returns one embedding per clip. We flatten it to one row of numbers and apply length normalization so clips have comparable embedding scale. The embedding network remains frozen.


In [ ]:
encoder = None

def load_encoder():
    global encoder
    if encoder is None:
        encoder = EncoderClassifier.from_hparams(
            source=MODEL_NAME,
            savedir=str(OUTPUT_DIR / 'pretrained_ecapa_tdnn'),
            run_opts={'device': DEVICE},
        )
    return encoder

def load_and_prepare_audio(path):
    waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    waveform, _ = librosa.effects.trim(waveform, top_db=30)
    waveform = waveform[:SAMPLE_RATE * MAX_SECONDS]
    if len(waveform) < SAMPLE_RATE:
        raise ValueError('Less than one second of speech')
    peak = np.max(np.abs(waveform))
    if peak > 0:
        waveform = waveform / peak
    return np.ascontiguousarray(waveform, dtype=np.float32)

def extract_ecapa_embedding(path):
    waveform = load_and_prepare_audio(path)
    signal = torch.from_numpy(waveform).unsqueeze(0).to(DEVICE)  # batch × samples
    with torch.inference_mode():
        embedding = load_encoder().encode_batch(signal)
    vector = embedding.detach().cpu().numpy().reshape(-1).astype(np.float32)
    if not np.isfinite(vector).all():
        raise ValueError('Non-finite embedding')
    norm = np.linalg.norm(vector)
    return vector / norm if norm > 1e-12 else vector


### Visual 1: what one clip looks like

The waveform shows amplitude over time. The log-Mel spectrogram shows how energy changes across frequency bands. This is an **illustration of the input speech**, not a dump of the model's internal features: SpeechBrain computes those inside the pretrained network. Quiet or clipped recordings are easier to notice here.


In [ ]:
example_path = recordings[0][0]
example_audio = load_and_prepare_audio(example_path)
mel = librosa.feature.melspectrogram(y=example_audio, sr=SAMPLE_RATE, n_mels=64,
                                     n_fft=400, hop_length=160)
log_mel = librosa.power_to_db(mel, ref=np.max)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), layout='constrained')
time_axis = np.arange(len(example_audio)) / SAMPLE_RATE
axes[0].plot(time_axis, example_audio, linewidth=0.6)
axes[0].set(xlabel='Time (s)', ylabel='Amplitude', title=f'Example waveform: {example_path.name}')
image = axes[1].imshow(log_mel, aspect='auto', origin='lower',
                       extent=[0, len(example_audio) / SAMPLE_RATE, 0, log_mel.shape[0]],
                       cmap='magma')
axes[1].set(xlabel='Time (s)', ylabel='Mel band', title='Illustrative log-Mel spectrogram')
fig.colorbar(image, ax=axes[1], label='dB relative to clip maximum')
fig.savefig(OUTPUT_DIR / 'ecapa_audio_example.png', dpi=150)
plt.show()


## 4. Extract once and cache the ECAPA-TDNN embeddings

Downloading the model and running inference are the expensive parts. The cache is reused when the selected paths, labels, speaker groups, file sizes, modification times and extraction settings still match. If you replace an audio file or change settings, the notebook extracts again. The ECAPA-TDNN embedding network remains frozen throughout.


In [ ]:
CACHE_PATH = OUTPUT_DIR / 'beginner_ecapa_embeddings.npz'
paths_expected = np.array([str(row[0]) for row in recordings])
labels_expected = np.array([row[1] for row in recordings], dtype=np.int64)
groups_expected = np.array([row[2] for row in recordings])
file_signatures = np.array([f'{row[0].stat().st_size}:{row[0].stat().st_mtime_ns}' for row in recordings])
settings = f'{MODEL_NAME}|sr={SAMPLE_RATE}|max_seconds={MAX_SECONDS}|trim_db=30|peak_normalize=1|length_normalize=1'

use_cache = False
if CACHE_PATH.exists():
    with np.load(CACHE_PATH, allow_pickle=False) as cached:
        use_cache = (str(cached['settings']) == settings
                     and np.array_equal(cached['paths'], paths_expected)
                     and np.array_equal(cached['labels'], labels_expected)
                     and np.array_equal(cached['groups'], groups_expected)
                     and np.array_equal(cached['file_signatures'], file_signatures))
        if use_cache:
            X = cached['X'].astype(np.float32)
            labels = cached['labels'].astype(np.int64)
            groups = cached['groups'].astype(str)
            paths = cached['paths'].astype(str)
            print('Loaded cached ECAPA-TDNN embeddings:', CACHE_PATH)

if not use_cache:
    vectors, kept_rows = [], []
    for number, row in enumerate(recordings, start=1):
        try:
            vectors.append(extract_ecapa_embedding(row[0]))
            kept_rows.append(row)
        except Exception as exc:
            print('Skipping', row[0].name, ':', exc)
        if number % 25 == 0 or number == len(recordings):
            print(f'Processed {number}/{len(recordings)}')
    if not vectors:
        raise RuntimeError('No ECAPA-TDNN embeddings were extracted; check model download and audio input')
    X = np.vstack(vectors)
    labels = np.array([row[1] for row in kept_rows], dtype=np.int64)
    groups = np.array([row[2] for row in kept_rows])
    paths = np.array([str(row[0]) for row in kept_rows])
    kept_signatures = np.array([f'{row[0].stat().st_size}:{row[0].stat().st_mtime_ns}' for row in kept_rows])
    if len(kept_rows) == len(recordings):
        np.savez_compressed(CACHE_PATH, X=X, labels=labels, groups=groups,
                            paths=paths, file_signatures=kept_signatures, settings=np.array(settings))
        print('Saved cache:', CACHE_PATH)
    else:
        print('Some clips failed; cache was not saved. Fix those clips before a full run.')

assert X.ndim == 2 and len(X) == len(labels) == len(groups) == len(paths)
assert np.isfinite(X).all()
print('X shape:', X.shape, '(clips × ECAPA-TDNN embedding dimensions)')


## 5. Make a speaker-disjoint split

The ECAPA-TDNN model was trained elsewhere on VoxCeleb and is never fitted to these clips. The **accent classifier** must still be trained and tested on different speakers. This split is approximately 80/20, depending on how many clips each speaker has.

A shared random seed does not guarantee the same split as another notebook if its selected clips differ. Save and reuse the exact indices for a formal E1/i-vector/x-vector/ECAPA-TDNN/Whisper comparison.


In [ ]:
speaker_counts = {label: len(set(groups[labels == label])) for label in (0, 1)}
if min(speaker_counts.values()) < 2:
    raise ValueError('Each class needs at least two speakers after extraction')
n_splits = min(5, min(speaker_counts.values()))
splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, labels, groups))
train_speakers, test_speakers = set(groups[train_idx]), set(groups[test_idx])
assert not (train_speakers & test_speakers)
assert set(labels[train_idx]) == set(labels[test_idx]) == {0, 1}
print('Training clips:', len(train_idx), '|', Counter(labels[train_idx]))
print('Test clips:', len(test_idx), '|', Counter(labels[test_idx]))
print('Speaker overlap:', len(train_speakers & test_speakers))


### Visual 2: a two-dimensional view of the ECAPA-TDNN embeddings

Each point below is **one clip**. PCA reduces the many ECAPA-TDNN embedding dimensions to two for display; it is fitted on **training clips only**, then applied to test clips. Color shows the folder label and shape shows whether the speaker was held out. Overlap in this projection does not prove the classifier cannot separate the full vectors; separation does not prove accent is the cause, because corpus and recording differences can also separate points.


In [ ]:
from sklearn.decomposition import PCA

projection = PCA(n_components=2, random_state=RANDOM_SEED)
projection.fit(X[train_idx])
points = projection.transform(X)
fig, ax = plt.subplots(figsize=(7, 5))
for subset_name, indices, marker in [('Train', train_idx, 'o'), ('Test', test_idx, '^')]:
    for label, color, class_name in [(0, 'tab:blue', 'Non-SG'), (1, 'tab:orange', 'Singapore')]:
        chosen = indices[labels[indices] == label]
        ax.scatter(points[chosen, 0], points[chosen, 1], c=color, marker=marker,
                   s=38 if subset_name == 'Train' else 75, alpha=0.75,
                   edgecolors='black' if subset_name == 'Test' else 'none',
                   label=f'{class_name} · {subset_name}')
ax.set(xlabel=f'PC1 ({projection.explained_variance_ratio_[0]:.0%} of training variance)',
       ylabel=f'PC2 ({projection.explained_variance_ratio_[1]:.0%} of training variance)',
       title='Pretrained ECAPA-TDNN embeddings projected with training-set PCA')
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ecapa_pca.png', dpi=150)
plt.show()


## 6. Train a small classifier on training ECAPA-TDNN embeddings

`StandardScaler` and logistic regression form one pipeline. Scaling is fitted **only on training embeddings**. The classifier learns an accent boundary; the pretrained ECAPA-TDNN network's weights do not change. This lets us ask whether speaker-trained representations contain useful information for this classification task.


In [ ]:
classifier = Pipeline([
    ('scale', StandardScaler()),
    ('head', LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000,
                                random_state=RANDOM_SEED)),
])
classifier.fit(X[train_idx], labels[train_idx])
print('Trained accent classifier on', len(train_idx), 'clips')


## 7. Evaluate on unseen speakers

Balanced accuracy and macro F1 weight both classes. ROC-AUC measures ranking across possible thresholds. The confusion matrix uses rows for **true** labels and columns for **predicted** labels, in `[non-Singapore, Singapore]` order.

These are **clip-level** scores. Some speakers may contribute more clips than others; for a stronger final report, also average probabilities by speaker and evaluate at speaker level.


In [ ]:
predictions = classifier.predict(X[test_idx])
probabilities = classifier.predict_proba(X[test_idx])[:, 1]
matrix = confusion_matrix(labels[test_idx], predictions, labels=[0, 1])
metrics = {
    'balanced_accuracy': float(balanced_accuracy_score(labels[test_idx], predictions)),
    'macro_f1': float(f1_score(labels[test_idx], predictions, average='macro')),
    'roc_auc': float(roc_auc_score(labels[test_idx], probabilities)),
    'confusion_matrix': matrix.tolist(),
}
print(json.dumps(metrics, indent=2))
print(classification_report(labels[test_idx], predictions, labels=[0, 1],
                            target_names=['Non-Singapore', 'Singapore'], zero_division=0))
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap='Blues')
for (row, col), count in np.ndenumerate(matrix):
    ax.text(col, row, str(count), ha='center', va='center')
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Non-SG', 'Singapore'], yticklabels=['Non-SG', 'Singapore'],
       xlabel='Predicted', ylabel='True', title='Held-out speakers')
fig.colorbar(image, ax=ax)
plt.show()


### Visual 3: scores for held-out clips

The vertical axis is the classifier's Singapore-folder score for each test clip. The dashed line is the `0.5` decision threshold. Colors are **true folder labels** and red rings mark mistakes. This reveals whether errors are near the boundary or confidently wrong. These scores are not guaranteed to be calibrated accent probabilities.


In [ ]:
test_labels = labels[test_idx]
plot_rng = np.random.default_rng(RANDOM_SEED)
x_jitter = test_labels + plot_rng.uniform(-0.12, 0.12, size=len(test_labels))
fig, ax = plt.subplots(figsize=(6, 4))
for label, color, name in [(0, 'tab:blue', 'Non-SG'), (1, 'tab:orange', 'Singapore')]:
    mask = test_labels == label
    ax.scatter(x_jitter[mask], probabilities[mask], c=color, alpha=0.75, s=42,
               label=f'True: {name}')
errors = predictions != test_labels
ax.scatter(x_jitter[errors], probabilities[errors], facecolors='none',
           edgecolors='red', linewidths=1.8, s=115, label='Wrong prediction')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1, label='Decision threshold')
ax.set(xlim=(-0.4, 1.4), ylim=(-0.03, 1.03), xticks=[0, 1],
       xticklabels=['Non-SG', 'Singapore'], xlabel='True folder label',
       ylabel='Singapore-folder score', title='Held-out clip scores')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'ecapa_test_scores.png', dpi=150)
plt.show()


## 8. Save the split, classifier and result

The `.npz` cache contains embeddings and clip metadata. The model file contains the **accent classifier only**; the pretrained SpeechBrain checkpoint is stored in `outputs/ecapa_tdnn/pretrained_ecapa_tdnn`. Load `joblib` files only from a source you trust.


In [ ]:
classifier_path = OUTPUT_DIR / 'beginner_ecapa_classifier.joblib'
split_path = OUTPUT_DIR / 'beginner_ecapa_split.npz'
report_path = OUTPUT_DIR / 'beginner_ecapa_report.json'
joblib.dump(classifier, classifier_path)
np.savez_compressed(split_path, paths=paths, labels=labels, groups=groups,
                    train_idx=train_idx, test_idx=test_idx)
report = {
    'embedding_model': MODEL_NAME,
    'label_meaning': {'0': 'non_singapore', '1': 'singapore'},
    'seed': RANDOM_SEED,
    'split': {'train_clips': len(train_idx), 'test_clips': len(test_idx),
              'train_speakers': len(train_speakers), 'test_speakers': len(test_speakers),
              'speaker_overlap': len(train_speakers & test_speakers)},
    'metrics': metrics,
}
report_path.write_text(json.dumps(report, indent=2) + '\n')
print('Saved:', classifier_path, split_path, report_path, sep='\n')


## What to try next

1. Run E1, i-vector, x-vector, ECAPA-TDNN and Whisper on the **same clips and saved speaker split**. Compare balanced accuracy, macro F1 and confusion matrices.
2. Add a speaker-disjoint validation set before tuning classifier settings or an ensemble weight. Keep the test set for the final result.
3. Evaluate both accent groups with matched microphones, prompts and recording conditions. MNSC-versus-Common-Voice performance may mostly measure corpus differences.

ECAPA-TDNN was trained for speaker recognition, so its embedding can encode speaker and microphone traits as well as accent. Inspecting its score on matched recordings is particularly important.
